# Model vs WRDS Volatility Surface

This notebook compares the downloaded WRDS volatility surface with the model-implied surface on the same points and visualizes the error surface.

In [1]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from scipy.optimize import brentq
from scipy.stats import norm
import torch
from scipy.optimize import brentq

from dataset import OptionsDataModule
from model import OptionNetModule
warnings.filterwarnings("ignore")

In [2]:
# --- Config ---
SURFACE_CSV = './data/108105_surface/2025_C_vol_surface.csv'
CHECKPOINT_PATH = './logs/my_experiment/version_63/checkpoints/epoch=34-step=48825.ckpt'
TRAIN_DATA_DIR = './data/108105'
CP_FLAG = 'C'
TARGET_DATE = None  # e.g. '2025-06-03'
RISK_FREE_RATE = 0.04
DEFAULT_VIX = None  # set float if your CSV has missing vix

def choose_device():
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'

DEVICE = choose_device()
DEVICE

'mps'

In [3]:
def black_scholes_price(cp_flag, s, k, t_years, r, sigma):
    if t_years <= 0.0 or sigma <= 0.0 or s <= 0.0 or k <= 0.0:
        return np.nan
    d1 = (np.log(s / k) + (r + 0.5 * sigma**2) * t_years) / (sigma * np.sqrt(t_years))
    d2 = d1 - sigma * np.sqrt(t_years)
    if cp_flag == 'P':
        return k * np.exp(-r * t_years) * norm.cdf(-d2) - s * norm.cdf(-d1)
    return s * norm.cdf(d1) - k * np.exp(-r * t_years) * norm.cdf(d2)

def implied_volatility(cp_flag, price, s, k, t_years, r):
    obj_func = lambda sigma: black_scholes_price(cp_flag, s, k, t_years, r, sigma) - price

    try:
        # brentq requires a bracket where the function changes sign
        return brentq(obj_func, a=1e-5, b=2.0, xtol=1e-6)
    except ValueError:
        # This happens if the price is physically impossible under BS (e.g., below intrinsic)
        print("IMPOSSIBLE PRICE UNDER BS")
        return np.nan

def expected_raw_feature_count(model):
    first_linear = model.model.model[0]
    return int(first_linear.in_features) + 1

def build_input_frame(df, raw_feature_count, default_vix=0.4):
    w = df.copy()
    if 'spot_price' not in w.columns:
        raise ValueError("CSV must contain 'spot_price'.")
    if 'impl_strike' not in w.columns:
        raise ValueError("CSV must contain 'impl_strike'.")
    if 'days' not in w.columns:
        raise ValueError("CSV must contain 'days'.")

    if 'vix' not in w.columns:
        w['vix'] = default_vix
    if w['vix'].isna().any():
        raise ValueError("Missing VIX values. Set DEFAULT_VIX or provide vix in CSV.")

    x = pd.DataFrame({
        'S': w['spot_price'].astype(float),
        'K': w['impl_strike'].astype(float),
        'T': w['days'].astype(float),
        'vix': w['vix'].astype(float),
    })

    if raw_feature_count == 4:
        return x

    if raw_feature_count == 9:
        for c in ['hv_10', 'hv_14', 'hv_30', 'hv_60', 'hv_91']:
            if c not in w.columns:
                w[c] = np.nan
            w[c] = w[c].astype(float)
            w[c] = w[c].fillna(w[c].median())
            w[c] = w[c].fillna(0.0)
            x[c] = w[c]
        return x

    raise ValueError(f'Unsupported raw feature count: {raw_feature_count}')

In [4]:
df = pd.read_csv(SURFACE_CSV)
df["date"] = pd.to_datetime(df["date"])
df = df[df["cp_flag"] == CP_FLAG].copy()
if df.empty:
    raise ValueError(f"No rows found for cp_flag={CP_FLAG}.")

target_date = pd.to_datetime(TARGET_DATE) if TARGET_DATE else df["date"].max()
daily = df[df["date"] == target_date].copy()
if daily.empty:
    raise ValueError(f"No rows found for date={target_date.date()} and cp_flag={CP_FLAG}.")

data_module = OptionsDataModule(TRAIN_DATA_DIR, batch_size=128)
data_module.setup("fit")

model = OptionNetModule.load_from_checkpoint(CHECKPOINT_PATH)
model.eval()

device = choose_device()
model.to(device)

raw_feature_count = expected_raw_feature_count(model)
features = build_input_frame(daily, raw_feature_count=raw_feature_count)

scaled_t_v = data_module.x_scaler.transform(features.loc[:, ["T", "vix"]])
features_scaled = features.copy()
features_scaled["T"] = scaled_t_v[:, 0]
features_scaled["vix"] = scaled_t_v[:, 1]

x_input = torch.tensor(features_scaled.values, dtype=torch.float32, device=device)

pred_scaled, greeks = model(x_input)
pred_prices = data_module.y_scaler.inverse_transform(pred_scaled.detach().cpu().numpy()).flatten()

t_years = daily["days"].astype(float).values / 365.0
cp_flags = daily["cp_flag"].astype(str).values
s_vals = daily["spot_price"].astype(float).values
k_vals = daily["impl_strike"].astype(float).values

model_iv = np.array(
    [
        implied_volatility(cp, px, s, k, t, RISK_FREE_RATE)
        for cp, px, s, k, t in zip(cp_flags, pred_prices, s_vals, k_vals, t_years, strict=True)
    ]
)

compared = daily.copy()
compared["model_price"] = pred_prices
compared["model_impl_volatility"] = model_iv
compared["iv_error"] = compared["model_impl_volatility"] - compared["impl_volatility"]
compared["abs_iv_error"] = compared["iv_error"].abs()

valid = compared[np.isfinite(compared["impl_volatility"]) & np.isfinite(compared["model_impl_volatility"])].copy()
if valid.empty:
    raise ValueError("No valid points after implied-vol inversion. Try another date or risk-free rate.")

rmse = float(np.sqrt(np.mean((valid["iv_error"]) ** 2)))
mae = float(np.mean(valid["abs_iv_error"]))
mean_abs_pct = float(np.mean(valid["abs_iv_error"] / valid["impl_volatility"].clip(lower=1e-8)) * 100.0)

metrics = {
        "secid": str(valid["secid"].iloc[0]),
        "cp_flag": CP_FLAG,
        "date": str(target_date.date()),
        "points_total": int(len(compared)),
        "points_valid": int(len(valid)),
        "rmse_iv": rmse,
        "mae_iv": mae,
        "mape_percent_iv": mean_abs_pct,
    }

for k, v in metrics.items():
    print(f"{k}: {v}")


Train: 357036 | Validation: 44629 | Test: 44630
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
IMPOSSIBLE PRICE UNDER BS
secid: 108105.0
cp_flag: C
date: 2025-08-29
points_total: 187
points_valid: 170
rmse_iv: 0.032357950209382154
mae_iv: 0.024724281614032475
mape_percent_iv: 16.152265051979455


In [7]:
def vol_surface_plot(
    df: pd.DataFrame,
    value_col: str,
    title: str,
    z_label: str,
) -> None:
    pivot = (
        df.pivot_table(index="days", columns="delta", values=value_col, aggfunc="mean")
        .sort_index()
        .sort_index(axis=1)
    )
    fig = go.Figure(
        data=[
            go.Surface(
                x=pivot.columns.values,
                y=pivot.index.values,
                z=pivot.values,
                colorscale="Viridis",
                hovertemplate=(
                    "Delta: %{x:.3f}<br>"
                    "Days: %{y:.0f}<br>"
                    f"{z_label}: %{{z:.5f}}<extra></extra>"
                ),
            )
        ]
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="Delta",
            yaxis_title="Days to Expiration",
            zaxis_title=z_label,
            aspectratio=dict(x=1, y=1, z=0.6),
        ),
        width=1000,
        height=700,
    )
    fig.show()


def price_surface_plot(
    df: pd.DataFrame,
    title: str,
    z_label: str = "Option Price",
    market_col: str = "impl_premium",
    model_col: str = "model_price",
) -> None:
    market_pivot = (
        df.pivot_table(index="days", columns="delta", values=market_col, aggfunc="mean")
        .sort_index()
        .sort_index(axis=1)
    )
    model_pivot = (
        df.pivot_table(index="days", columns="delta", values=model_col, aggfunc="mean")
        .sort_index()
        .sort_index(axis=1)
    )

    fig = go.Figure(
        data=[
            go.Surface(
                name="Market Price",
                x=market_pivot.columns.values,
                y=market_pivot.index.values,
                z=market_pivot.values,
                colorscale="Viridis",
                opacity=0.9,
                showscale=True,
                colorbar=dict(title="Market"),
                hovertemplate=(
                    "Surface: Market<br>"
                    "Delta: %{x:.3f}<br>"
                    "Days: %{y:.0f}<br>"
                    f"{z_label}: %{{z:.5f}}<extra></extra>"
                ),
            ),
            go.Surface(
                name="Model Price",
                x=model_pivot.columns.values,
                y=model_pivot.index.values,
                z=model_pivot.values,
                colorscale="Reds",
                opacity=0.65,
                showscale=True,
                colorbar=dict(title="Model", x=1.08),
                hovertemplate=(
                    "Surface: Model<br>"
                    "Delta: %{x:.3f}<br>"
                    "Days: %{y:.0f}<br>"
                    f"{z_label}: %{{z:.5f}}<extra></extra>"
                ),
            ),
        ]
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="Delta",
            yaxis_title="Days to Expiration",
            zaxis_title=z_label,
            aspectratio=dict(x=1, y=1, z=0.6),
        ),
        width=1000,
        height=700,
        legend=dict(x=0.01, y=0.99),
    )
    fig.show()


price_surface_plot(valid, f"Market vs Model Price Surface ({target_date.date()})", "Option Price")

#vol_surface_plot(valid, 'impl_volatility', f'Model vs Market IV Surface ({target_date.date()})', 'Market IV')
#vol_surface_plot(valid, 'model_impl_volatility', f'Model IV Surface ({target_date.date()})', 'Model IV')
#vol_surface_plot(valid, 'iv_error', f'Model - Market IV Error Surface ({target_date.date()})', 'IV Error')

